# Driver Price Acceptance Analysis

Notebook nay phan tich synthetic data cho bai toan gia xe cong nghe, tap trung vao tai xe. Hai feature chinh la `rain` va `rush_hour`. Muc tieu la du doan phu phi, sau do danh gia tac dong cua gia den kha nang tai xe nhan chuyen.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

sns.set_theme(style="whitegrid")
DATA_PATH = Path("../data/driver_synthetic_trips.csv")
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.describe(include="all")

In [ ]:
df[["rain", "rush_hour", "driver_accept"]].mean().rename(
    {
        "rain": "Ti le chuyen khi troi mua",
        "rush_hour": "Ti le chuyen trong gio cao diem",
        "driver_accept": "Ti le tai xe nhan chuyen",
    }
)

In [ ]:
price_summary = (
    df.groupby(["rain", "rush_hour"], as_index=False)
    .agg(
        avg_base_price=("base_price", "mean"),
        avg_delta_price=("delta_price", "mean"),
        avg_price=("price", "mean"),
        acceptance_rate=("driver_accept", "mean"),
        n_trips=("trip_id", "count"),
    )
)
price_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=df, x="rain", y="delta_price", ax=axes[0], errorbar="sd")
axes[0].set_title("Phu phi trung binh theo trang thai mua")
axes[0].set_xlabel("Rain")
axes[0].set_ylabel("Delta price")

sns.barplot(data=df, x="rush_hour", y="delta_price", ax=axes[1], errorbar="sd")
axes[1].set_title("Phu phi trung binh theo gio cao diem")
axes[1].set_xlabel("Rush hour")
axes[1].set_ylabel("Delta price")
plt.tight_layout()

In [ ]:
linear_model = LinearRegression()
linear_model.fit(df[["rain", "rush_hour"]], df["delta_price"])

linear_coefficients = pd.DataFrame(
    {
        "feature": ["intercept", "rain", "rush_hour"],
        "coefficient": [linear_model.intercept_, *linear_model.coef_],
    }
)
linear_coefficients

In [ ]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(df[["price", "rain", "rush_hour"]], df["driver_accept"])

predicted_accept = logistic_model.predict(df[["price", "rain", "rush_hour"]])
predicted_prob = logistic_model.predict_proba(df[["price", "rain", "rush_hour"]])[:, 1]

logistic_coefficients = pd.DataFrame(
    {
        "feature": ["intercept", "price", "rain", "rush_hour"],
        "coefficient": [logistic_model.intercept_[0], *logistic_model.coef_[0]],
    }
)

metrics = {
    "accuracy": accuracy_score(df["driver_accept"], predicted_accept),
    "roc_auc": roc_auc_score(df["driver_accept"], predicted_prob),
}

logistic_coefficients, metrics

In [ ]:
df_bins = df.assign(price_bin=pd.qcut(df["price"], q=6))
acceptance_by_price = (
    df_bins.groupby("price_bin", observed=True)
    .agg(
        avg_price=("price", "mean"),
        acceptance_rate=("driver_accept", "mean"),
        n_trips=("trip_id", "count"),
    )
    .reset_index()
)
acceptance_by_price

In [ ]:
plt.figure(figsize=(8, 4))
sns.lineplot(data=acceptance_by_price, x="avg_price", y="acceptance_rate", marker="o")
plt.title("Ti le tai xe nhan chuyen theo muc gia")
plt.xlabel("Gia trung binh")
plt.ylabel("Ti le nhan chuyen")
plt.ylim(0, 1)
plt.tight_layout()

In [ ]:
rain_effect = linear_coefficients.loc[linear_coefficients["feature"] == "rain", "coefficient"].iloc[0]
rush_effect = linear_coefficients.loc[linear_coefficients["feature"] == "rush_hour", "coefficient"].iloc[0]
price_effect = logistic_coefficients.loc[logistic_coefficients["feature"] == "price", "coefficient"].iloc[0]

conclusion = f"""
Ket luan:
- Khi troi mua, phu phi du doan tang trung binh khoang {rain_effect:,.0f} VND.
- Trong gio cao diem, phu phi du doan tang trung binh khoang {rush_effect:,.0f} VND.
- He so price trong mo hinh Logistic Regression la {price_effect:.6f} va lon hon 0.
- Dieu nay phu hop voi gia dinh: gia chuyen xe cao hon lam xac suat tai xe nhan chuyen tang len.
"""
print(conclusion)